# 1. Download data from https://www.kaggle.com/c/DontGetKicked . Design train/validation/test split.

Все эти разбиения были проделана в ml4, там же и была предобрабоотка пропущенных значений и все encoder.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from category_encoders import CountEncoder
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('./data/training.csv', parse_dates=['PurchDate'])

In [3]:
df = df.sort_values('PurchDate')
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
32367,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
32384,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
32385,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
32386,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
32387,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


In [4]:
train_valid_df, test_df = train_test_split(df, test_size=0.33333, shuffle=False)
train_df, valid_df = train_test_split(train_valid_df, test_size=0.5, shuffle=False)

In [5]:
X_train = train_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_train = train_df['IsBadBuy']

X_valid = valid_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_valid = valid_df['IsBadBuy']

X_test = test_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_test = test_df['IsBadBuy']

In [6]:
cat_selector = make_column_selector(dtype_include=['object', 'category'])
cat_columns = cat_selector(train_df)
cat_columns

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'PRIMEUNIT',
 'AUCGUART',
 'VNST']

In [7]:
def remove_nan_values(X_train, X_valid, X_test):
     # убираем 2 столбца с пропущенными значениями
    X_train.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)
    X_valid.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)
    X_test.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)

    features_top = ['Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType']
    features_mean = ['MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
    
    for feature in features_top:
        base = X_train[feature].value_counts().iloc[0]

        X_train.loc[X_train[feature].isna(), feature] = base
        X_valid.loc[X_valid[feature].isna(), feature] = base
        X_test.loc[X_test[feature].isna(), feature] = base

    for feature in features_mean:
        base = X_train[feature].mean()

        X_train.loc[X_train[feature].isna(), feature] = base
        X_valid.loc[X_valid[feature].isna(), feature] = base
        X_test.loc[X_test[feature].isna(), feature] = base
    
    return X_train, X_valid, X_test

In [8]:
X_train, X_valid, X_test = remove_nan_values(X_train, X_valid, X_test)

In [9]:
y_valid.drop(index=X_valid[X_valid['Size'].isna()].index, inplace=True)
X_valid.drop(index=X_valid[X_valid['Size'].isna()].index, inplace=True)
X_valid.isna().sum().sum()

np.int64(0)

In [10]:
categorical_columns = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'VNST']

In [11]:
for col in categorical_columns:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [12]:
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]
numeric_columns

['RefId',
 'VehYear',
 'VehicleAge',
 'WheelTypeID',
 'VehOdo',
 'MMRAcquisitionAuctionAveragePrice',
 'MMRAcquisitionAuctionCleanPrice',
 'MMRAcquisitionRetailAveragePrice',
 'MMRAcquisitonRetailCleanPrice',
 'MMRCurrentAuctionAveragePrice',
 'MMRCurrentAuctionCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'MMRCurrentRetailCleanPrice',
 'BYRNO',
 'VNZIP1',
 'VehBCost',
 'IsOnlineSale',
 'WarrantyCost']

In [13]:
X_valid.loc[X_valid.Transmission == 'Manual', 'Transmission'] = 'MANUAL'

In [14]:
categorical_columns.remove('Make')
categorical_columns.remove('Model')
categorical_columns.remove('Trim')
categorical_columns.remove('SubModel')
categorical_columns.remove('VNST')

transformers = [
    ('num', 'passthrough', numeric_columns),
    ('cat', OneHotEncoder(), categorical_columns),
    ('count_scaled', CountEncoder(), ['Make', 'Model', 'Trim', 'SubModel', 'VNST'])
]

ct = ColumnTransformer(transformers, remainder='drop')
X_train_transform = ct.fit_transform(X_train)
X_valid_transform = ct.transform(X_valid)
X_test_transform = ct.transform(X_test)

In [15]:
pd.DataFrame(X_train_transform, columns=ct.get_feature_names_out())

,num__RefId,num__VehYear,num__VehicleAge,num__WheelTypeID,num__VehOdo,num__MMRAcquisitionAuctionAveragePrice,num__MMRAcquisitionAuctionCleanPrice,num__MMRAcquisitionRetailAveragePrice,num__MMRAcquisitonRetailCleanPrice,num__MMRCurrentAuctionAveragePrice,...,cat__Size_VAN,cat__TopThreeAmericanName_CHRYSLER,cat__TopThreeAmericanName_FORD,cat__TopThreeAmericanName_GM,cat__TopThreeAmericanName_OTHER,count_scaled__Make,count_scaled__Model,count_scaled__Trim,count_scaled__SubModel,count_scaled__VNST
0,32389.0,2007.0,2.0,2.0,78541.0,7261.0,8857.0,8342.0,10066.0,8709.0,...,0.0,1.0,0.0,0.0,0.0,2774.0,99.0,4540.0,284.0,2016.0
1,32406.0,2005.0,4.0,1.0,37676.0,4409.0,5734.0,5262.0,6693.0,4908.0,...,1.0,0.0,1.0,0.0,0.0,4264.0,309.0,265.0,46.0,2016.0
2,32407.0,2004.0,5.0,2.0,71680.0,3098.0,4061.0,3846.0,4886.0,3397.0,...,0.0,1.0,0.0,0.0,0.0,4784.0,250.0,3432.0,1427.0,2016.0
3,32408.0,2006.0,3.0,1.0,69456.0,8530.0,9883.0,9712.0,11174.0,9202.0,...,0.0,0.0,0.0,1.0,0.0,5659.0,55.0,2988.0,177.0,2016.0
4,32409.0,2004.0,5.0,1.0,66530.0,3094.0,4230.0,3842.0,5068.0,3369.0,...,0.0,0.0,1.0,0.0,0.0,4264.0,955.0,265.0,58.0,2016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24322,56894.0,2005.0,4.0,2.0,70531.0,3150.0,4011.0,3902.0,4832.0,3148.0,...,0.0,0.0,0.0,1.0,0.0,5659.0,68.0,2988.0,4903.0,2554.0
24323,56893.0,2008.0,1.0,2.0,64301.0,8607.0,9350.0,9796.0,10598.0,7965.0,...,0.0,1.0,0.0,0.0,0.0,4784.0,276.0,3432.0,4903.0,2554.0
24324,56892.0,2008.0,1.0,2.0,64524.0,8607.0,9350.0,9796.0,10598.0,8568.0,...,0.0,1.0,0.0,0.0,0.0,4784.0,276.0,3432.0,4903.0,2554.0
24325,15848.0,2006.0,3.0,2.0,78217.0,8702.0,10517.0,9898.0,11858.0,7590.0,...,0.0,0.0,0.0,0.0,1.0,198.0,23.0,69.0,61.0,4562.0


Все пропущенные значения предобработаны и применены енкодеры для категориальных признаков. Нормализация не производилась, так как для моделей с деревьями она не нужна.

# 2. Create a Python class for DEcision Tree Classifier and Decosoin Tree Regressor (MSE loss)

класс Node - узла дерева, который хранит данные, условия разделения и указатели на правого и левого потомков

In [16]:
class Node:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.left_child = None
        self.right_child = None
        self.split_feature = None
        self.split_value = None

    def set_left_child(self, pointer):
        self.left_child = pointer

    def set_right_child(self, pointer):
        self.right_child = pointer

    def get_Gini(self):
        probability_y_0 = (self.y == 0).sum() / len(self.y)
        probability_y_1 = 1 - probability_y_0

        gini_impurity = probability_y_0*(1 - probability_y_0) + probability_y_1*(1 - probability_y_1)

        return gini_impurity
    
    def get_standard_devation(self):
        return np.std(self.y)

    def get_X(self):
        return self.X

    def get_y(self):
        return self.y

In [17]:
len(X_train_transform)

24327

класс DecisionTreeClassifier - то есть строить деревья по информативности gini и решает задачу бинароной классификации

In [18]:
class myDesicionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=2, max_features=None, random_state=21):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state

    def _compute_gini(self, y):
         if len(y) == 0: return 0
         P = (y == 1).sum() / len(y)
         return 2 * P * (1 - P)

    def find_best_split(self, node):
        X = node.get_X()
        y = node.get_y()

        if node.get_Gini() == 0:
            return None, None

        optimal_feature = None
        optimal_value = None
        best_score = 0

        np.random.seed(self.random_state)

        if self.max_features is not None:
            feature_indices = np.random.choice(X.shape[1], self.max_features, replace=False)
        else:
            feature_indices = range(X.shape[1])

        for i in feature_indices:
            feature_value = X[:, i]
            n_unique = len(np.unique(feature_value))

            if n_unique > 100:
                percentiles = np.linspace(0, 100, 100)
                candidate_values = np.percentile(feature_value, percentiles)
            else:
                candidate_values = np.unique(feature_value)


            for value in candidate_values:
                left_mask = X[:,i] <= value
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]
                
                if (len(y_left) < self.min_samples_split) or (len(y_right) < self.min_samples_split):
                     continue

                left_gini = self._compute_gini(y_left)
                right_gini = self._compute_gini(y_right)

                score = len(X) * node.get_Gini() - len(y_left) * left_gini - len(y_right) * right_gini

                if score > best_score:
                    best_score = score
                    optimal_feature = i
                    optimal_value = value
        
        return optimal_feature, optimal_value
    
    def prune_tree(self, node):
        node.X = None
        
        if node.left_child:
            self.prune_tree(node.left_child)
        if node.right_child:
            self.prune_tree(node.right_child)
        
    def fit(self, X_train, y_train):
        self.root = Node(X_train, y_train)
        self.make_split(self.root, 2)

        self.prune_tree(self.root)
        

    def make_split(self, current_node, level):
        if level >= self.max_depth:
            return
        
        if len(current_node.y) < self.min_samples_split:
            return
         
        split_feature, split_value = self.find_best_split(current_node)

        if split_feature is not None:
            left_mask = current_node.X[:,split_feature] <= split_value
            right_mask = current_node.X[:,split_feature] > split_value

            left_child = Node(current_node.X[left_mask], current_node.y[left_mask])
            right_child = Node(current_node.X[right_mask], current_node.y[right_mask])

            current_node.split_feature = split_feature
            current_node.split_value = split_value

            current_node.set_left_child(left_child)
            current_node.set_right_child(right_child)

            self.make_split(current_node.left_child, level + 1)
            self.make_split(current_node.right_child, level + 1)

    def get_probability(self, sample, current_node):
        if current_node.split_feature is None:
            return np.mean(current_node.y)
        
        if sample[current_node.split_feature] <= current_node.split_value:
            return self.get_probability(sample, current_node.left_child)
        else:
            return self.get_probability(sample, current_node.right_child)

    
    def predict_proba(self, X):
        answer = []
        for sample in X:
            answer.append(self.get_probability(sample, self.root))
        
        return np.array(answer)

    def predict(self, X, threshold=0.5):
        predict = (self.predict_proba(X) > 0.5).astype(int)
        return predict

In [19]:
class myDesicionTreeRegression:
    def __init__(self, max_depth=10, min_samples_split=2, max_features=20, random_state=21):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state

    def _compute_standard_devation(self, y):
        if len(y) == 0: return 0
        return np.std(y)

    def find_best_split(self, node):
        X = node.get_X()
        y = node.get_y()

        if node.get_standard_devation() == 0:
            return None, None

        optimal_feature = None
        optimal_value = None
        best_score = 0

        np.random.seed(self.random_state)
        indices = np.arange(X.shape[1])
        np.random.shuffle(indices)

        for i in indices[:self.max_features]:
            feature_value = X[:, i]
            n_unique = len(np.unique(feature_value))

            if n_unique > 100:
                percentiles = np.linspace(0, 100, 100)
                candidate_values = np.percentile(feature_value, percentiles)
            else:
                candidate_values = np.unique(feature_value)

            for value in candidate_values:
                left_mask = X[:,i] <= value
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]
                
                if (len(y_left) < self.min_samples_split) or (len(y_right) < self.min_samples_split):
                     continue

                left_standard_devation = self._compute_standard_devation(y_left)
                right_standard_devation = self._compute_standard_devation(y_right)

                score = len(X) * node.get_standard_devation() - len(y_left) * left_standard_devation - len(y_right) * right_standard_devation

                if score > best_score:
                    best_score = score
                    optimal_feature = i
                    optimal_value = value
        
        return optimal_feature, optimal_value
        
    def prune_tree(self, node):
        node.X = None
        
        if node.left_child:
            self.prune_tree(node.left_child)
        if node.right_child:
            self.prune_tree(node.right_child)
        
    def fit(self, X_train, y_train):
        self.root = Node(X_train, y_train)
        self.make_split(self.root, 2)

        self.prune_tree(self.root)
        

    def make_split(self, current_node, level):
        if level >= self.max_depth:
            return
        
        if len(current_node.y) < self.min_samples_split:
            return
         
        split_feature, split_value = self.find_best_split(current_node)

        if split_feature is not None:
            left_mask = current_node.X[:,split_feature] <= split_value
            right_mask = current_node.X[:,split_feature] > split_value

            left_child = Node(current_node.X[left_mask], current_node.y[left_mask])
            right_child = Node(current_node.X[right_mask], current_node.y[right_mask])

            current_node.split_feature = split_feature
            current_node.split_value = split_value

            current_node.set_left_child(left_child)
            current_node.set_right_child(right_child)

            self.make_split(current_node.left_child, level + 1)
            self.make_split(current_node.right_child, level + 1)

    def predict(self, X):
        predictions = []
        for sample in X:
            node = self.root
            while node.left_child or node.right_child:
                if sample[node.split_feature] <= node.split_value:
                    node = node.left_child
                else:
                    node = node.right_child
            predictions.append(np.mean(node.y))
        return np.array(predictions)

ExtraRandomizedTree - класс, где логика как для классификации, только мы не ищем раздедение не по всем признакам, а только по рандомной части из них, и при этом мы не перебираем все возможные значения для разделения, или не берём квантили, а берем некоторое рандомно количество разлиичным примеров, и уже среди них ищем лучшее разделение.

In [20]:
class myExtraRandomizedTree:
    def __init__(self, max_depth=10, min_samples_split=2, max_features=20, random_state=21):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state

    def _compute_gini(self, y):
         if len(y) == 0: return 0
         P = (y == 1).sum() / len(y)
         return 2 * P * (1 - P)

    def find_best_split(self, node):
        X = node.get_X()
        y = node.get_y()

        if node.get_Gini() == 0:
            return None, None

        optimal_feature = None
        optimal_value = None
        best_score = 0

        np.random.seed(self.random_state)
        indices = np.arange(X.shape[1])
        np.random.shuffle(indices)

        for i in indices[:self.max_features]:
            feature_value = X[:, i]
            n_unique = len(np.unique(feature_value))

            if n_unique > 20:
                candidate_values = np.random.choice(feature_value, size=20, replace=False)
            else:
                candidate_values = np.random.choice(feature_value, size=n_unique // 2, replace=False)
            
            for value in candidate_values:
                left_mask = X[:,i] <= value
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]
                
                if (len(y_left) < self.min_samples_split) or (len(y_right) < self.min_samples_split):
                     continue

                left_gini = self._compute_gini(y_left)
                right_gini = self._compute_gini(y_right)

                score = len(X) * node.get_Gini() - len(y_left) * left_gini - len(y_right) * right_gini

                if score > best_score:
                    best_score = score
                    optimal_feature = i
                    optimal_value = value
        
        return optimal_feature, optimal_value
        
    def prune_tree(self, node):
        node.X = None
        
        if node.left_child:
            self.prune_tree(node.left_child)
        if node.right_child:
            self.prune_tree(node.right_child)
        
    def fit(self, X_train, y_train):
        self.root = Node(X_train, y_train)
        self.make_split(self.root, 2)

        self.prune_tree(self.root)
        

    def make_split(self, current_node, level):
        if level >= self.max_depth:
            return
        
        if len(current_node.y) < self.min_samples_split:
            return
         
        split_feature, split_value = self.find_best_split(current_node)

        if split_feature is not None:
            left_mask = current_node.X[:,split_feature] <= split_value
            right_mask = current_node.X[:,split_feature] > split_value

            left_child = Node(current_node.X[left_mask], current_node.y[left_mask])
            right_child = Node(current_node.X[right_mask], current_node.y[right_mask])

            current_node.split_feature = split_feature
            current_node.split_value = split_value

            current_node.set_left_child(left_child)
            current_node.set_right_child(right_child)

            self.make_split(current_node.left_child, level + 1)
            self.make_split(current_node.right_child, level + 1)

    def get_probability(self, sample, current_node):
        if current_node.split_feature is None:
            return np.mean(current_node.y)
        
        if sample[current_node.split_feature] <= current_node.split_value:
            return self.get_probability(sample, current_node.left_child)
        else:
            return self.get_probability(sample, current_node.right_child)

    
    def predict_proba(self, X):
        answer = [self.get_probability(sample, self.root) for sample in X]
        
        return np.array(answer)

    def predict(self, X, threshold=0.5):
        predict = (self.predict_proba(X) > 0.5).astype(int)
        return np.array(predict)

# 3. Wuth your DicisionTree module, you must obtain a Gini score of at least 0.1 on the validation dataset.

In [21]:
def gini_score(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

In [22]:
my_tree_classifier = myDesicionTreeClassifier(max_depth=5)
my_tree_classifier.fit(X_train_transform, y_train) 

In [23]:
my_tree_gini = gini_score(y_valid, my_tree_classifier.predict_proba(X_valid_transform))

print(f'My Gini for myDecisionTreeClassifier:  {my_tree_gini}')

My Gini for myDecisionTreeClassifier:  0.4258553278712276


# 4. Use sklearn's DecisionTreeClassifier and checl ots perfomance on the validation dataset. Is it better then your module? If so, why ?

In [24]:
tree_classifier = DecisionTreeClassifier(max_depth=5)
tree_classifier.fit(X_train_transform, y_train)
tree_gini = gini_score(y_valid, tree_classifier.predict_proba(X_valid_transform)[:,1])

In [25]:
print(f'Gini for sklearn decisionTreeClassifier:  {tree_gini}')
print(f'My Gini for myDecisionTreeClassifier:  {my_tree_gini}')

Gini for sklearn decisionTreeClassifier:  0.4524264510073599
My Gini for myDecisionTreeClassifier:  0.4258553278712276


У sklearn DecisionTreeClassifer значения gini score лучше, чем моя реализация. Скорее всего это связано с более оптимизирвоанными и лучшими поисками разделения, а также большим количество параметров, для обучения. То есть моя модель неплохая, но в сравнении намного меньше гибкости параметров. 

Также при обучении в моей модели, если уникальных значений более 100, то я брал квантили, а не перебирал все занчения, так как перебирать полностью слишком долго. А sklearn за счёт более оптимизированных методов может находить разделения лучше моих.

# 5. Implement the RandomForestClassifier  and check its perfomance. You have to improve the result of a single tree and get at least 0.15 Gini score on validation dataset. Be able to set a fixed random seed.

In [26]:
class myRandomForestClassifier:
    def __init__(self, n_estimators=100, random_state=21, max_depth=15, max_features='sqrt'):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
    
    def fit(self, X, y):
        if self.max_features == 'sqrt':
            max_features = int(np.sqrt(X.shape[1]))
        else:
            raise print(f'Unknown value of parameter max_features:  {self.max_features}???')
        
        np.random.seed(self.random_state)
        tree_random_state = self.random_state
        indices = np.arange(len(X))
        yy = np.array(y)
        
        for i in tqdm(range(self.n_estimators), desc=f'Fitting  forest'):

            tree = myDesicionTreeClassifier(max_depth=self.max_depth, max_features=max_features, random_state=tree_random_state)
            tree_random_state += 1
        
            subsampling_indices = np.random.choice(indices, size=len(X), replace=True)

            X_subsampling = X[subsampling_indices]
            y_subsampling = yy[subsampling_indices]

            tree.fit(X_subsampling, y_subsampling)

            self.trees.append(tree)
    
    def predict_proba(self, X):
        votes = [tree.predict_proba(X) for tree in self.trees]

        return np.mean(votes, axis=0)
    
    def predict(self, X, threshold=0.5):
        predictions = (self.predict_proba(X) > threshold).astype(int)

        return np.array(predictions)

In [27]:
my_forest = myRandomForestClassifier(max_depth=10)
my_forest.fit(X_train_transform, y_train)

Fitting  forest: 100%|██████████| 100/100 [00:58<00:00,  1.72it/s]


In [28]:
forest_gini_ = gini_score(y_valid, my_forest.predict_proba(X_valid_transform))

print(f'Gini score for my RandomForestClassifier:  {forest_gini_}')

Gini score for my RandomForestClassifier:  0.4554602354851165


# 6. Use your DecisionTree design class for GBDT classfier. 

This class must have max_depth, number_of_trees and max_features attributes.

You must compute the gradient of the binary cross-entropy loss function and implement incremential learning: train the next tree using the results of the previous trees.

Формула антиградиента для binary cross_entropy loss function: y/p - (1-y)/(1-p)

In [29]:
class myGBDT:
    def __init__(self, max_depth=3, number_of_trees=100, max_features=9, random_state=21, learning_rate=0.1):
        self.max_depth=max_depth
        self.number_of_trees=number_of_trees
        self.max_features=max_features
        self.random_state=random_state
        self.learning_rate = learning_rate
        self.trees = []

    def fit(self, X, y):
        self.base_predict = np.mean(y)
        current_pred = np.full(len(y), self.base_predict)

        for i in tqdm(range(self.number_of_trees), desc='Fitting GBDT'):
            tree = myDesicionTreeRegression(max_depth=self.max_depth, max_features=self.max_features, random_state=self.random_state + i)
            
            target = y/current_pred - (1-y) / (1-current_pred)
            tree.fit(X, target)

            tree_pred = tree.predict(X)
            current_pred += self.learning_rate * tree_pred
            current_pred = np.clip(current_pred, 1e-15, 1-1e-15)

            self.trees.append(tree)
            
    def predict_proba(self, X):
        predict = np.full(X.shape[0], self.base_predict)
        
        tree_preds = [tree.predict(X) for tree in self.trees]

        total_correction = np.sum(tree_preds, axis=0) * self.learning_rate
    
        return predict + total_correction
    
    def predict(self, X, threshold=0.5):
        predictions = (self.predict_proba(X) >= threshold).astype(int)

        return predictions

In [30]:
my_gbdt = myGBDT(learning_rate=0.01)
my_gbdt.fit(X_train_transform, y_train)

Fitting GBDT: 100%|██████████| 100/100 [01:10<00:00,  1.43it/s]


In [31]:
gbdt_gini = gini_score(y_valid, my_gbdt.predict_proba(X_valid_transform))

print(f'Gini score for my GBDT classifier:  {gbdt_gini}')

Gini score for my GBDT classifier:  0.4610974943849675


# 7. Use LightGBM, Catboost and XGBoost for fitting on a traning set and prediction on a validation set.

Rewiew the documentation of the libraries and fine_tune the algorithms for the task. Note key differences between each implementation. Analyze special features of each algorithm (how does 'categorical feature' work in Catboost, what is DART mode in XGBoost)? Which GBDT model gives the best result ? Can you explain why?

### CatBoost

Алгоритм, разработанный Яндексом в 2017 году. Это метод градиентного бустинга на решающих деревевьях. Особенности:
- может обрабатывать категриальные признакки. То есть в него встроенные инструменты обработки категориальных признаков, что позволяет не заниматься самостоятельной предобработкой данных. Данные предобрабатываются по разным критериям и весьма эффективно. Можно просто указать параметр cat_features, в котором передать какие признаки являются категорильными, чтобы аглоритм их обработал.
- способ построения деревьев отличается от других методов, а именно строятся бинарные сбалансированнные деревеься, и также на каждом уровне дерева, во всех узлах этого уровня одинаковый критерий деления. Данный способ построения деревьев минимизирует переобучение. То есть не только за счёт ограничения глубины дерева или других способ, а своей архитектурой метода.
- процесс обучения и предсказания весьма быстрый. Конечно для LightGBM быстрее обучение, но это и очевидно, так как сам метод облегчённый.
- имеет множество встроенных методов, которые упрощяют работу, помимо втроенных способ обработка категорильаных признаков, модель сама моежет побирать занчения learning_rate по данным
- также может работать с текстовыми признаками (параметр test_features метода fit)

In [32]:
cat_columns = cat_selector(X_train)
cat_columns

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'VNST']

In [33]:
model_CatBoost = CatBoostClassifier(random_state=21)
model_CatBoost.fit(X_train, y_train, cat_features=cat_columns)

Learning rate set to 0.040253
0:	learn: 0.6554291	total: 192ms	remaining: 3m 12s
1:	learn: 0.6215789	total: 237ms	remaining: 1m 58s
2:	learn: 0.5922339	total: 263ms	remaining: 1m 27s
3:	learn: 0.5638864	total: 304ms	remaining: 1m 15s
4:	learn: 0.5377464	total: 349ms	remaining: 1m 9s
5:	learn: 0.5154086	total: 392ms	remaining: 1m 4s
6:	learn: 0.4948314	total: 434ms	remaining: 1m 1s
7:	learn: 0.4774266	total: 477ms	remaining: 59.1s
8:	learn: 0.4617348	total: 506ms	remaining: 55.8s
9:	learn: 0.4471563	total: 539ms	remaining: 53.4s
10:	learn: 0.4343732	total: 568ms	remaining: 51s
11:	learn: 0.4219472	total: 607ms	remaining: 50s
12:	learn: 0.4098117	total: 649ms	remaining: 49.3s
13:	learn: 0.3993896	total: 693ms	remaining: 48.8s
14:	learn: 0.3901518	total: 729ms	remaining: 47.9s
15:	learn: 0.3819824	total: 773ms	remaining: 47.5s
16:	learn: 0.3752290	total: 789ms	remaining: 45.6s
17:	learn: 0.3683712	total: 831ms	remaining: 45.4s
18:	learn: 0.3617401	total: 873ms	remaining: 45.1s
19:	learn: 

In [34]:
cat_boost_gini = gini_score(y_valid, model_CatBoost.predict_proba(X_valid)[:,1])

print(f'Gini score for CatBoost:  {cat_boost_gini}')

Gini score for CatBoost:  0.48921298668817137


## LightGBM

Главная особенность метода в том, что самый быстрый по сравнению с другими. Очень эффективный для больших данных и нетребовательный к ресурсам. Больше всего времени в алгоритмах бустинга занимает построение деревьев, поэтому в LightGBM решили строить деревья быстрее, за счёт другой структры, уменьшения количества наблюдений и уменьшения количества признаков, по которым ищется критерий разделения.

- построение деревьев происходит по листам, то есть ищет лучшее рабиение в листе, из-за чего деревья могут получаться несимметричными и глубокими. Однако такой метод построения гораздо быстрее остальных.
- выбираются наблюдения с большим градиентом, так как они сильнее влияют, а с маленьким градиентом наблюдения не сильно помогут в обучении, зато заметно повысится скорость обучения. Отбор наблюдений происходит при помощи метода GOSS
- при помощи метода EFB вместо k-признаков строят p-связок, пирчём p <= k. Данный алгоритм строить связки таким образом, чтобы вделить важные связанные признаки, чтобы не потерять в качестве модели. И так как таких связок меньше, чем признаков, обучение можели происходит быстрее.
- linear_tree - будут использвоаться не обычные деревья, где ответы в листьях константы, а ответом будет линейная модель на основе признаков.

In [35]:
lightGBM_model = LGBMClassifier(random_state=21, n_jobs=-1, linear_tree=True, learning_rate=0.01)
lightGBM_model.fit(X_train_transform, y_train)

[LightGBM] [Info] Number of positive: 2794, number of negative: 21533
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000923 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3646
[LightGBM] [Info] Number of data points in the train set: 24327, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114852 -> initscore=-2.042112
[LightGBM] [Info] Start training from score -2.042112


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.01
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [36]:
lightGBM_gini = gini_score(y_valid, lightGBM_model.predict_proba(X_valid_transform)[:,1])

print(f'Gini score for LightGBM model:  {lightGBM_gini}')

Gini score for LightGBM model:  0.46082956632788563


c:\Users\User\anaconda3\envs\ml_projects\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## XGBoost

- основное отличие - это втроенная регуляризация в функию ошибок. Штраф за сложность дерева. Регуляризация весов в лситьях и l1, l2 регуляризцая. Позволяет бороться в переобучением и снижать его риск.
- при поиске разделений, перебираются не все возможные уникальные значения, а происходит разбиение по квантилям, и уже по кванитлям ищется разделение.
- автоматичеки обрабатывает пропущенные занчеия.
- деревья строятся по уровням, то есть дерево будет симметричным, однако на одном уровне в узлах могут быть различные критерии разделения
- DART - специльнай режим, при котором при добавлении нового дерева происходит случайное удаление ранее обученных деревьев. При обучении, послдение деревья теряют обощающие способности, и подстраиваются уже под конкретные примеры. Удаление некторых деревеьев позвоаляет созранять обощающие способности.

In [37]:
xgb_model = XGBClassifier(max_depth=3, booster='dart', n_jobs=-1, random_state=21, n_estimators=100)
xgb_model.fit(X_train_transform, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,'dart'
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [38]:
xgb_gini = gini_score(y_valid, xgb_model.predict_proba(X_valid_transform)[:,1])

print(f'Gini score for XGBoost:  {xgb_gini}')

Gini score for XGBoost:  0.47356877841834355


## Сравнение и выводы по всем 3 моделям

In [39]:
print(f'Gini score for CatBoost:  {cat_boost_gini}')
print(f'Gini score for XGBoost:  {xgb_gini}')
print(f'Gini score for LightGBM model:  {lightGBM_gini}')

Gini score for CatBoost:  0.48921298668817137
Gini score for XGBoost:  0.47356877841834355
Gini score for LightGBM model:  0.46082956632788563


Модель CatBoost даёт лучшие значения метрики. Связано это с тем, что:
- при обучении подавались данные исходные X_train, в которых оставались категорильные признаки, которые модель сама кодировала. Для остальных же моделей использовались X_train_tranform, и возможно обработка категорильаных признаков в CatBoost лучше, чем моя обработка, вследствии чего значения лучше.
- CatBoost также сам выбирал значения learning_rate при помощи своих эвристик, когда в случае в остлаьными моделями приходилось подбирать самому. Возможно при более тщательном подборе гиперпаметров, другие модели могли бы показать лучшие значение метрик. То есть для дургих моделей возможно нужна более тонкая настройка гиперпарамтеров.

# 8. Take the best model and estimate its performance on the test dataset.

Is your model overfitting or not? Explain.

### Лучшее значение получилось у CatBoost

In [40]:
cat_boost_gini_train = gini_score(y_train, model_CatBoost.predict_proba(X_train)[:,1])
cat_boost_gini_valid = gini_score(y_valid, model_CatBoost.predict_proba(X_valid)[:,1])
cat_boost_gini_test = gini_score(y_test, model_CatBoost.predict_proba(X_test)[:,1])

print(f'CatBoost Gini score:')
print(f'Train:  {cat_boost_gini_train}')
print(f'Valid:  {cat_boost_gini_valid}')
print(f'Test:  {cat_boost_gini_test}')

CatBoost Gini score:
Train:  0.6941217490385567
Valid:  0.48921298668817137
Test:  0.4983360279430875


Можем видеть, что значения Gini на тестовой выборке даже выше, чем на валидационной. Это гооворит о том, что модель не переобучилась и дотстаочно хорошо работате на незнакомых данных, на которых смотрим метрику только сейчас.

# 9*. Bonus part. Implementj the ExtraTreesClassifier  and check its performance. 

You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.

ExtraTreesClassifier - это как randomForest, только мы не передаем подвыборку, я передаём все данные, но при этом деревья не обычные, а ExtraRandomizeTree, в котором мы не перебираем уникальные значения или квантили, а берём n рандомно различных значений и среди них ищем лучшее разделение

In [41]:
class myExtraTreesClassifier:
    def __init__(self, n_estimators=100, random_state=21, max_depth=15, max_features='sqrt'):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
    
    def fit(self, X, y):
        if self.max_features == 'sqrt':
            max_features = int(np.sqrt(X.shape[1]))
        else:
            raise ValueError(f'Unknown value of parameter max_features:  {self.max_features}???')
        
        tree_random_state = self.random_state
        
        for i in tqdm(range(self.n_estimators), desc=f'Fitting  forest'):

            tree = myExtraRandomizedTree(max_depth=self.max_depth, max_features=max_features, random_state=tree_random_state)
            tree_random_state += 1
    
            tree.fit(X, y)

            self.trees.append(tree)
    
    def predict_proba(self, X):
        votes = [tree.predict_proba(X) for tree in self.trees]

        return np.mean(votes, axis=0)
    
    def predict(self, X, threshold=0.5):
        predictions = (self.predict_proba(X) > threshold).astype(int)

        return np.array(predictions)

In [42]:
my_ExtraTrees = myExtraTreesClassifier(n_estimators=50, max_depth=10)
my_ExtraTrees.fit(X_train_transform, y_train)

Fitting  forest: 100%|██████████| 50/50 [01:03<00:00,  1.27s/it]


In [43]:
my_ExtraTrees_gini = gini_score(y_valid, my_ExtraTrees.predict_proba(X_valid_transform))

print(f'Gini score for my ExtraTreesClassifier:  {my_ExtraTrees_gini}')


Gini score for my ExtraTreesClassifier:  0.44523931112390924


In [44]:
print(f'Gini score for my RandomForestClassifier:  {forest_gini_}')

Gini score for my RandomForestClassifier:  0.4554602354851165


Значений немного хуже, однако значения выбирались ранодмно, а не перебом. Хотя в мое реализации есть ограничени, и перебор идёт по квантилям, но если сравнивать с классичсеким подходом, то ExtraTrees очень хорошо справляется с задачей и существенно быстрее прямого перебора. 
* Плюсы:
    - такой подход с рандомных поиском раздееления позволяет бороться с переобучением. 
    - Также данный метод быстрее классического, что весьма важно на больших наборах данных. 
    - лучше обобщает данные, устойчивее к выборсам и шуму в данных.
- Минусы
    - меньшая точность, так как случаные пороги могут пропустить оптимальное разделение
    - требует больше деревьев для достижения той же точности, что и RandomForest
    - сложнее интерпретировать